# Markovovy řetězce Monte Carlo (MCMC)

*Nad rámec skripta — aplikace.*

---

Kurz nabídl tři způsoby, jak vyrobit náhodnou veličinu s danou hustotou:

| metoda | co potřebuje | cvičení |
|---|---|---|
| inverzní transformace | umět spočítat $F^{-1}$ | 4 |
| zamítací metoda | obálku $M g \ge f$, ze které umíme generovat | 6 |
| superpozice | rozklad na jednodušší složky | 5 |

Každá z nich na něčem stojí. Co když **nemáme nic z toho**? $F^{-1}$ neexistuje v uzavřeném tvaru, rozumná obálka se nenajde a ve vyšší dimenzi zamítací metoda stejně zkolabuje — ve cvičení 8 jsme viděli, že v kouli dimenze 20 projde jeden bod ze čtyřiceti milionů.

**Metropolisův algoritmus** problém otočí. Nevyrábí nezávislé vzorky, ale postaví **náhodnou procházku**, která se v dlouhodobém průměru zdržuje přesně tam, kde má cílová hustota velkou hodnotu. Zaplatíme za to tím, že vzorky jsou **závislé** — a přesně proto jsme v sešitu `extra/ch01_05_kdy_se_mc_vyplati` řešili, jak u závislých vzorků počítat chybu.

V sešitu uvidíte:

- Metropolisův algoritmus a proč funguje (detailní rovnováha)
- **volbu kroku**: příliš malý i příliš velký je špatně, a rozdíl je padesátinásobný
- **rozjezd** (burn-in) a chybové úsečky u závislých vzorků
- kde MCMC vyhrává: rovnoměrný bod v kouli dimenze 20, kde zamítací metoda nemá šanci

In [ ]:
import math
import random

import matplotlib.pyplot as plt

random.seed(1)  # opakovatelné výsledky


# --- cílová hustota: směs dvou normálních (nenormovaná by stačila taky) ---
VAHY = (0.35, 0.65)
STREDY = (-2.0, 2.5)
ODCHYLKY = (0.8, 1.2)


def cil(x):
    """Hustota směsi. Pro Metropolise stačí i nenormovaná — konstanta se vykrátí."""
    return sum(w * math.exp(-((x - m) / s) ** 2 / 2) / (s * math.sqrt(2 * math.pi))
               for w, m, s in zip(VAHY, STREDY, ODCHYLKY))


EX = sum(w * m for w, m in zip(VAHY, STREDY))
EX2 = sum(w * (m * m + s * s) for w, m, s in zip(VAHY, STREDY, ODCHYLKY))
DX = EX2 - EX**2
print(f"cílové rozdělení má přesně: E X = {EX:.5f},  D X = {DX:.5f}")

## 1. Metropolisův algoritmus

Myšlenka: postavíme náhodnou procházku $X_0, X_1, X_2, \dots$ tak, aby se v ní dlouhodobě zdržovala v poměru daném cílovou hustotou $f$.

Jeden krok ze stavu $x$:

1. **Navrhni** $y = x + \varepsilon$, kde $\varepsilon \sim N(0, \sigma^2)$ (symetrický návrh).
2. Spočítej **poměr** $\displaystyle \alpha = \frac{f(y)}{f(x)}$.
3. Je-li $\alpha \ge 1$, **přijmi** $y$. Jinak přijmi $y$ s pravděpodobností $\alpha$, a když nepřijmeš, **zůstaň na $x$** (a zapiš $x$ znovu).

Dvě věci stojí za zdůraznění:

- Potřebujeme jen **poměr** $f(y)/f(x)$, takže normalizační konstanta se vykrátí. To je ten hlavní důvod popularity MCMC: v bayesovské statistice normalizační konstantu obvykle neumíme spočítat.
- Zamítnutí **neznamená opakovat losování**, jako u zamítací metody. Znamená zapsat tutéž hodnotu podruhé. Proto jsou vzorky závislé.

Bod 3 je přesně Metropolisovo kritérium ze simulovaného žíhání (`extra/ch09_stochasticke_algoritmy`), jen tam sloužilo k hledání minima a tady ke generování.

### Proč to funguje

Chceme, aby $f$ byla **stacionárním rozdělením** řetězce. Stačí k tomu **detailní rovnováha**: pro všechna $x, y$

$$f(x)\,P(x \to y) = f(y)\,P(y \to x).$$

Ověřme ji. Pro symetrický návrh je hustota návrhu $q(x \to y) = q(y \to x)$, takže pro $f(y) < f(x)$

$$f(x)\,P(x\to y) = f(x)\,q\,\frac{f(y)}{f(x)} = f(y)\,q = f(y)\,P(y \to x),$$

protože opačný krok se přijímá vždy ($\alpha \ge 1$). Rovnost platí, a stacionární rozdělení je tedy $f$.

In [ ]:
def metropolis(hustota, n, krok, x0=0.0):
    """Metropolisův algoritmus se symetrickým normálním návrhem.

    Vrací (vzorky, podíl přijatých). Vzorky jsou ZÁVISLÉ — viz oddíl 3.
    """
    x = x0
    fx = hustota(x)
    out = []
    prijato = 0
    for _ in range(n):
        y = x + random.gauss(0.0, krok)
        fy = hustota(y)
        if fy >= fx or random.random() < fy / fx:
            x, fx = y, fy
            prijato += 1
        out.append(x)
    return out, prijato / n


N = 200_000
vzorky, prijeti = metropolis(cil, N, krok=3.0)
vzorky = vzorky[N // 10:]                    # zahodíme rozjezd, viz oddíl 3

m = sum(vzorky) / len(vzorky)
v = sum((x - m) ** 2 for x in vzorky) / (len(vzorky) - 1)
print(f"přijato {prijeti:.1%} návrhů")
print(f"E X: {m:.5f}   (přesně {EX:.5f})")
print(f"D X: {v:.5f}   (přesně {DX:.5f})")
assert abs(m - EX) < 0.05 and abs(v - DX) < 0.2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), gridspec_kw={"width_ratios": [1, 1.3]})

ax = axes[0]
w = 12 / 80
ax.hist(vzorky, bins=80, range=(-6, 6), weights=[1 / (len(vzorky) * w)] * len(vzorky),
        alpha=0.55, label="Metropolis")
gr = [-6 + 12 * i / 400 for i in range(401)]
ax.plot(gr, [cil(x) for x in gr], "r", lw=1.6, label="cílová hustota")
ax.set_xlabel("$x$")
ax.set_ylabel("hustota")
ax.set_title("Řetězec navštěvuje stavy v poměru hustoty", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(range(1500), vzorky[:1500], lw=0.7)
for m_, s_ in zip(STREDY, ODCHYLKY):
    ax.axhline(m_, color="r", ls="--", lw=0.8)
ax.set_xlabel("krok řetězce")
ax.set_ylabel("$X_k$")
ax.set_title("Průběh: řetězec přeskakuje mezi oběma vrcholy", fontsize=10)

plt.tight_layout()
plt.show()

Histogram sedí na cílovou hustotu, přestože jsme nikdy nepočítali $F^{-1}$ ani nehledali obálku.

Pravý graf ukazuje, co řetězec dělá: chvíli se drží u jednoho vrcholu, pak přeskočí k druhému. Právě tyhle přeskoky musí nastat dost často, jinak by poměr obou vrcholů vyšel špatně — a jak často nastávají, závisí na **velikosti kroku**.

## 2. Volba kroku: ani malý, ani velký

Krok $\sigma$ je jediný parametr a je s ním potíž na obě strany:

- **příliš malý** → skoro všechno se přijme, ale řetězec se sotva hne a trvá věčnost, než projde celé rozdělení,
- **příliš velký** → návrhy padají daleko do oblastí s malou hustotou, skoro všechno se zamítne a řetězec stojí na místě.

Obojí vypadá v průběhu podobně (řetězec se nehýbe) a obojí poznáme podle **autokorelačního času** $\tau$ ze sešitu `extra/ch01_05_kdy_se_mc_vyplati`: z $n$ závislých vzorků máme informaci jen zhruba $n/\tau$ nezávislých.

In [ ]:
def autokorelacni_cas(xs, maxlag=400):
    """τ = 1 + 2 Σ ρ_l, sečteno dokud autokorelace neklesne pod šum."""
    n = len(xs)
    m = sum(xs) / n
    var = sum((v - m) ** 2 for v in xs) / n
    tau = 1.0
    for l in range(1, maxlag):
        c = sum((xs[i] - m) * (xs[i + l] - m) for i in range(n - l)) / ((n - l) * var)
        if c < 0.02:
            break
        tau += 2 * c
    return tau


print(f"{'krok σ':>8s} {'přijato':>9s} {'E X':>9s} {'D X':>9s} {'τ':>8s} {'efektivních vzorků':>20s}")
kroky = (0.2, 1.0, 3.0, 8.0, 25.0)
tau_list, ess_list, ar_list = [], [], []
for krok in kroky:
    xs, ar = metropolis(cil, N, krok)
    xs = xs[N // 10:]
    m = sum(xs) / len(xs)
    v = sum((q - m) ** 2 for q in xs) / (len(xs) - 1)
    t = autokorelacni_cas(xs[:40_000])
    tau_list.append(t); ess_list.append(len(xs) / t); ar_list.append(ar)
    print(f"{krok:>8.1f} {ar:>9.1%} {m:>9.4f} {v:>9.4f} {t:>8.1f} {len(xs) / t:>20,.0f}".replace(",", " "))

print(f"\nMezi nejlepším a nejhorším krokem je {max(ess_list) / min(ess_list):.0f}násobný rozdíl v počtu")
print(f"efektivních vzorků. Optimum leží u přijetí kolem 40 %.")
print(f"\nVšimněte si prvního řádku: při σ = 0.2 vyjde E X {abs(EX - 0.74) / EX:.0%} vedle. Není to vychýlení")
print(f"algoritmu — ten je nestranný asymptoticky pro každý krok — ale τ ≈ {tau_list[0]:.0f} znamená, že")
print(f"z {len(xs):,} vzorků máme informaci jen asi o {ess_list[0]:.0f} nezávislých. Na to je rozdíl dvou".replace(",", " "))
print(f"vrcholů prostě moc velký. Málo informace vypadá úplně stejně jako špatný výsledek.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))

ax = axes[0]
for krok, barva in ((0.2, "tab:red"), (3.0, "tab:green"), (25.0, "tab:purple")):
    xs, ar = metropolis(cil, 3000, krok)
    ax.plot(xs[:1200], lw=0.7, alpha=0.85, color=barva, label=f"σ = {krok} (přijetí {ar:.0%})")
ax.set_xlabel("krok řetězce")
ax.set_ylabel("$X_k$")
ax.set_title("Malý krok se plazí, velký stojí na místě", fontsize=10)
ax.legend(fontsize=8, loc="upper right")

ax = axes[1]
ax.loglog(kroky, ess_list, "o-")
for k_, e_, a_ in zip(kroky, ess_list, ar_list):
    ax.annotate(f"{a_:.0%}", (k_, e_), textcoords="offset points", xytext=(0, -14),
                ha="center", fontsize=8)
ax.set_xlabel("velikost kroku σ (log)")
ax.set_ylabel("efektivních vzorků (log)")
ax.set_title("Optimum uprostřed; popisky = podíl přijatých", fontsize=10)

plt.tight_layout()
plt.show()

## 3. Rozjezd a chybové úsečky

Dvě praktické věci, bez kterých se MCMC nedá použít.

### Rozjezd (burn-in)

Řetězec musíme někde začít a ten začátek obvykle nemá s cílovým rozdělením nic společného. Chvíli trvá, než se tam dostane, a těch prvních pár tisíc kroků se musí **zahodit** — jinak by nám do průměru vstupovaly stavy, kde cílová hustota je prakticky nulová.

Je to přesně ten **přechodový jev**, který jsme viděli u fronty v sešitu `extra/ch07_hromadna_obsluha`: simulace startuje z netypického stavu a začátek běhu se zahazuje.

### Chybová úsečka

Obvyklý vzorec $\hat\sigma/\sqrt{n}$ platí **jen pro nezávislé vzorky**. Vzorky z MCMC nezávislé nejsou, takže by nám dal příliš úzký interval a falešnou jistotu. Správná odpověď je **průměr po blocích** ze sešitu `extra/ch01_05_kdy_se_mc_vyplati`.

In [ ]:
# --- rozjezd: start daleko od cíle ---
start_daleko = 40.0
retezec, _ = metropolis(cil, 4000, krok=3.0, x0=start_daleko)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
ax = axes[0]
ax.plot(retezec, lw=0.8)
ax.axhline(EX, color="r", ls="--", lw=1, label=f"$E X$ = {EX:.2f}")
ax.axvspan(0, 500, color="tab:red", alpha=0.12, label="rozjezd (zahodit)")
ax.set_xlabel("krok řetězce")
ax.set_ylabel("$X_k$")
ax.set_title(f"Start v $x_0$ = {start_daleko:.0f}: chvíli trvá, než řetězec dorazí", fontsize=10)
ax.legend(fontsize=8)

# --- chybové úsečky: naivní vs po blocích ---
def chyba_naivni(xs):
    n = len(xs)
    m = sum(xs) / n
    return math.sqrt(sum((v - m) ** 2 for v in xs) / (n - 1) / n)


def chyba_po_blocich(xs, B=2000):
    nb = len(xs) // B
    bloky = [sum(xs[i * B:(i + 1) * B]) / B for i in range(nb)]
    m = sum(bloky) / nb
    return math.sqrt(sum((v - m) ** 2 for v in bloky) / (nb - 1) / nb)


# skutečné kolísání odhadu: 60 nezávislých řetězců
odhady = []
for _ in range(60):
    xs_, _ = metropolis(cil, 40_000, krok=3.0, x0=random.gauss(0, 3))
    odhady.append(sum(xs_[4000:]) / len(xs_[4000:]))
m_o = sum(odhady) / len(odhady)
skutecna = math.sqrt(sum((v - m_o) ** 2 for v in odhady) / (len(odhady) - 1))

xs_jeden, _ = metropolis(cil, 40_000, krok=3.0)
xs_jeden = xs_jeden[4000:]
print(f"{'metoda':>26s} {'chyba odhadu E X':>18s}")
print(f"{'naivní σ/√n':>26s} {chyba_naivni(xs_jeden):>18.5f}")
print(f"{'po blocích (B = 2000)':>26s} {chyba_po_blocich(xs_jeden):>18.5f}")
print(f"{'skutečná (60 řetězců)':>26s} {skutecna:>18.5f}")
print(f"\nNaivní vzorec podhodnocuje {skutecna / chyba_naivni(xs_jeden):.1f}krát. Po blocích to sedí.")

ax = axes[1]
ax.hist(odhady, bins=18, alpha=0.6, label="odhady E X z 60 řetězců")
ax.axvline(EX, color="r", ls="--", label=f"přesně {EX:.3f}")
ax.errorbar([m_o], [3], xerr=[chyba_naivni(xs_jeden)], fmt="o", color="tab:red",
            capsize=4, label="naivní úsečka (moc úzká)")
ax.errorbar([m_o], [1.5], xerr=[chyba_po_blocich(xs_jeden)], fmt="o", color="tab:green",
            capsize=4, label="po blocích")
ax.set_xlabel("odhad $E X$")
ax.set_ylabel("počet řetězců")
ax.set_title("Naivní úsečka slibuje přesnost, kterou nemá", fontsize=10)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Rozptyl odhadů ze šedesáti nezávislých řetězců je ta pravda, se kterou se obě úsečky porovnávají. Naivní vzorec dává úsečku několikanásobně užší, než jaké je skutečné kolísání — slíbil by přesnost, kterou odhad nemá. Průměr po blocích sedí.

## 4. Kde MCMC opravdu vyhrává: vyšší dimenze

Dosud to byla jen jiná cesta k témuž cíli — směs dvou normálních umíme vygenerovat i superpozicí (cvičení 5), a to přesně a nezávisle. Skutečný důvod, proč MCMC existuje, je vidět až ve vyšší dimenzi.

Vezměme úlohu ze cvičení 8: **rovnoměrný bod v jednotkové kouli dimenze $d$**. Zamítací metoda z krychle má pravděpodobnost přijetí $V_d / 2^d$, která s dimenzí padá propastně — v dimenzi 20 je potřeba čtyřicet milionů pokusů na jeden bod.

Metropolis má přitom úlohu snadnou: cílová hustota je konstantní uvnitř koule a nulová venku, takže **návrh se přijme právě tehdy, když padne dovnitř koule**. Krok $\sigma$ si zvolíme tak, aby to vycházelo dost často.

Kontrola: pro rovnoměrný bod v kouli je

$$E\,\|X\|^2 = \frac{d}{d+2},$$

což jsme odvodili ve cvičení 8.

In [ ]:
def koule_metropolis(d, n, krok, x0=None):
    """Rovnoměrný bod v jednotkové kouli dimenze d. Vrací (||x||² v každém kroku, přijetí)."""
    x = [0.0] * d if x0 is None else list(x0)
    r2 = sum(q * q for q in x)
    out, prijato = [], 0
    for _ in range(n):
        y = [xi + random.gauss(0.0, krok) for xi in x]
        ry2 = sum(q * q for q in y)
        if ry2 < 1.0:                      # hustota je uvnitř konstantní, venku nulová
            x, r2 = y, ry2
            prijato += 1
        out.append(r2)
    return out, prijato / n


def objem_koule(d):
    return math.pi ** (d / 2) / math.gamma(d / 2 + 1)


N2 = 200_000
print(f"{'d':>3s} {'krok':>6s} {'přijetí':>9s} {'E||X||² MCMC':>14s} {'přesně d/(d+2)':>15s} "
      f"{'zamítací: pokusů na bod':>24s}")
dims, pokusy = [], []
for d, krok in ((2, 0.50), (5, 0.35), (10, 0.22), (20, 0.14)):
    out, ar = koule_metropolis(d, N2, krok)
    out = out[N2 // 5:]
    m = sum(out) / len(out)
    presne = d / (d + 2)
    zamitaci = 2**d / objem_koule(d)
    dims.append(d); pokusy.append(zamitaci)
    popis = format(zamitaci, ",.0f").replace(",", " ")
    print(f"{d:>3d} {krok:>6.2f} {ar:>9.1%} {m:>14.5f} {presne:>15.5f} {popis:>24s}")
    assert abs(m - presne) < 0.01, d

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
dd = list(range(2, 26))
ax.semilogy(dd, [2**d / objem_koule(d) for d in dd], "r-o", ms=4,
            label="zamítací metoda: pokusů na jeden bod")
ax.axhline(1.0, color="tab:green", ls="--",
           label="Metropolis: ~1 návrh na krok (přijetí neklesá k nule tak rychle)")
ax.set_xlabel("dimenze $d$")
ax.set_ylabel("pokusů na jeden bod (log)")
ax.set_title("Proč MCMC existuje: cena zamítací metody roste exponenciálně", fontsize=11)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("Metropolis ovšem nedává NEZÁVISLÉ body — za snadné generování platíme")
print("autokorelací, takže i tady je potřeba počítat τ a chybu po blocích.")

Červená křivka je cena zamítací metody, zelená zhruba cena Metropolise. V nízké dimenzi je zamítací metoda lepší — je jednoduchá a dává **nezávislé** vzorky. Od dimenze zhruba deset se to otáčí a kolem dvacítky už zamítací metoda nemá smysl.

Pozor ale na poctivé srovnání: Metropolisovy body **nejsou nezávislé**. Jeden krok je sice levný, ale aby se získal jeden *efektivní* vzorek, je potřeba zhruba $\tau$ kroků. Zelená čára je tedy optimistická — i tak ale zůstává o mnoho řádů pod červenou.

## Shrnutí

| Pojem | Co si pamatovat |
|-------|-----------------|
| Metropolis | navrhni $y$, přijmi s pravděpodobností $\min(1, f(y)/f(x))$, jinak zůstaň |
| Normalizační konstanta | nepotřebná — vykrátí se v poměru. Hlavní důvod popularity |
| Detailní rovnováha | $f(x)P(x\to y) = f(y)P(y\to x)$ ⟹ $f$ je stacionární rozdělení |
| Volba kroku | malý = plazí se, velký = stojí; optimum u přijetí kolem 40 % |
| Autokorelační čas $\tau$ | z $n$ vzorků je informace jen o $n/\tau$ nezávislých |
| Rozjezd | začátek řetězce zahodit — stejně jako přechodový jev u fronty |
| Chybová úsečka | naivní $\hat\sigma/\sqrt n$ **podhodnocuje**; použít průměr po blocích |
| Kdy sáhnout po MCMC | když selže inverze i obálka, nebo ve vyšší dimenzi |

Hlavní myšlenka: MCMC vymění **nezávislost za použitelnost**. Vzorky jsou horší (závislé, s rozjezdem, s laditelným krokem), ale dostaneme je i tam, kde ostatní metody nemají šanci. A protože jsou závislé, přenáší se veškerá péče o chybové úsečky ze sešitu o závislých realizacích.

### Zkuste si

1. Nahraďte symetrický návrh nesymetrickým (třeba $y \sim \operatorname{Exp}$ kolem $x$). Pak je potřeba plný Metropolisův–Hastingsův poměr $\frac{f(y)q(y\to x)}{f(x)q(x\to y)}$. Ověřte, že bez opravy vyjde rozdělení špatně.
2. Zvětšete vzdálenost obou vrcholů směsi (třeba na $\pm 10$). Jak musí vzrůst $N$, aby odhad $E X$ pořád seděl? Co to říká o vícevrcholových rozděleních?
3. Napište **Gibbsův** vzorkovač pro dvourozměrné normální rozdělení s korelací $\rho$ (střídavě losujte $x \mid y$ a $y \mid x$) a změřte $\tau$ jako funkci $\rho$.
4. Ověřte detailní rovnováhu numericky: na jemné mřížce spočítejte $f(x)P(x\to y)$ a $f(y)P(y\to x)$ a porovnejte.
5. Pro kouli v dimenzi 20 změřte $\tau$ a spočítejte skutečný počet kroků na jeden efektivní vzorek. Je Metropolis pořád lepší než zamítací metoda?
6. Zkuste jako cíl Cauchyho rozdělení (cvičení 4). Funguje Metropolis i pro těžké chvosty? Co to udělá s $\tau$?